# آزمایش ایدهٔ جابه‌جایی مخصوص هر سلول

سلول‌ها را **یکی‌یکی از بالا به پایین** اجرا کن. ابتدا داده را می‌بینیم، سپس مدل اصلی را آموزش می‌دهیم و نتیجه‌اش را می‌بینیم. بعد همان مدل را ثابت نگه می‌داریم و فقط شبکهٔ ایدهٔ تو را آموزش می‌دهیم.

در بالای نوت‌بوک از **Select Kernel** محیط `ali_env` را انتخاب کن. فعلاً **Run All نزن**؛ هر مرحله را با هم بررسی می‌کنیم.


In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'cellshift' / 'run.py').exists()), None)
if ROOT is None:
    raise RuntimeError('نوت‌بوک را از داخل پوشهٔ پروژه باز کن.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from cellshift.data import load_gbm, load_simulated, normalized, split
from cellshift.run import subset, scenarios, metrics, dump
from cellshift.model import Backbone, bandwidth_from_training, train_adapter
from cellshift.upstream import create_scdisinfact_dataset, scdisinfact, REVISION
print('پایتون انتخاب‌شده:', sys.executable)
print('GPU قابل استفاده:', torch.cuda.is_available())
print('پوشهٔ پروژه:', ROOT)


## ۱. دریافت دادهٔ مقاله

اگر دانلود در ترمینال هنوز در حال اجراست، صبر کن تمام شود و **این سلول را اجرا نکن**. اگر `Ready: gbm` دیده‌ای، این مرحله را رد کن و به مرحلهٔ ۲ برو. برای دانلود از خود نوت‌بوک، سلول زیر را اجرا کن. فایل کامل و سالم دوباره دانلود نمی‌شود.


In [ ]:
import subprocess

def run_visible(arguments):
    # همان پایتونی استفاده می‌شود که برای kernel انتخاب کرده‌ای.
    process = subprocess.Popen([sys.executable, '-u', '-m', *arguments], cwd=ROOT,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise RuntimeError('اجرای مرحله ناموفق بود؛ متن خروجی بالا را بررسی کن.')
    except BaseException:
        process.terminate()
        process.wait()
        raise

run_visible(['cellshift.download', 'gbm'])


## ۲. ببینیم چه داده‌ای داریم

هر ردیف یک سلول است و هر ستونِ ماتریس، یک ژن. جدول زیر اطلاعات سلول‌ها را نشان می‌دهد؛ نه خودِ شمارش ژن‌ها. عددهای این مرحله نتیجهٔ مدل نیستند.


In [ ]:
data = load_gbm(ROOT / 'data')
print(f'{data.counts.shape[0]:,} سلول و {data.counts.shape[1]:,} ژن')
display(data.meta[['batch', 'sample_id', 'treatment', 'cell_type']].head(10))
populations = pd.crosstab(data.meta.cell_type, data.meta.treatment)
display(populations)
populations.plot.bar(figsize=(9, 4), title='Number of cells by cell type and treatment')
plt.ylabel('Cells'); plt.tight_layout(); plt.show()


## ۳. جداکردن آموزش، اعتبارسنجی و تست

نمونهٔ `PW034-705` تست است؛ مدل نباید آن را هنگام آموزش ببیند. از بقیه، بخشی برای انتخاب بهترین شبکه کنار گذاشته می‌شود. هیچ‌یک از این دو بخش وارد آموزش مدل پایه نمی‌شوند.


In [ ]:
SEED = 0
manifest, _ = split(data, 'gbm', SEED, min_cells=50)
display(pd.crosstab(manifest.role, manifest.treatment))
display(manifest.groupby(['role', 'batch', 'cell_type']).size().rename('cells').to_frame())
training = subset(data, manifest.role.eq('train'))
validation = subset(data, manifest.role.eq('val'))
assert not set(training.meta.index) & set(manifest.index[manifest.role.eq('test')])
print('سلول‌های تست از آموزش جدا هستند.')


## ۴. تنظیم اجرای مدل اصلی

فعلاً تنظیمات اصلی مقاله را نگه می‌داریم: ۱۰۰ دور آموزش. اگر GPU شناسایی نشود، آموزش خودکار شروع نمی‌شود؛ بررسی داده‌های بالا همچنان قابل انجام است. اجرای CPU را فقط با انتخاب آگاهانهٔ `ALLOW_CPU = True` فعال کن؛ ممکن است طولانی شود.

هر بار اجرای این سلول پوشهٔ خروجی تازه‌ای می‌سازد. تا پایان یک آزمایش، این سلول را دوباره اجرا نکن.


In [ ]:
from datetime import datetime
import random
ALLOW_CPU = False
BASE_EPOCHS = 100
ADAPTER_EPOCHS = 100
STEPS = 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = ROOT / 'runs_notebook' / datetime.now().strftime('%Y%m%d_%H%M%S_%f') / 'gbm' / f'seed_{SEED}'
OUT.mkdir(parents=True)
manifest.to_csv(OUT / 'split.csv')
dump(OUT / 'config.json', {'seed': SEED, 'revision': REVISION, 'base_epochs': BASE_EPOCHS,
                          'adapter_epochs': ADAPTER_EPOCHS, 'steps': STEPS, 'device': str(DEVICE)})
print('اجرا روی:', DEVICE)
print('ذخیرهٔ خروجی:', OUT)


## ۵. آموزش مدل اصلی مقاله

در این مرحله هنوز شبکهٔ پیشنهادی تو وجود ندارد. چاپ lossها نشان می‌دهد آموزش جلو می‌رود. اگر این سلول کامل شد، برای دیدن نتیجه به مرحلهٔ ۶ برو.


In [ ]:
if DEVICE.type == 'cpu' and not ALLOW_CPU:
    raise RuntimeError('GPU در این kernel قابل استفاده نیست. قبل از آموزش باید آن را بررسی کنیم؛ برای اجرای عمدی CPU، ALLOW_CPU را True کن.')
if (OUT / 'backbone.pt').exists():
    raise RuntimeError('مدل این اجرا قبلاً ذخیره شده؛ برای اجرای جدید از مرحلهٔ ۴ شروع کن.')
torch.set_num_threads(4)
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
dictionary = create_scdisinfact_dataset(training.counts, training.meta.copy(),
    condition_key=data.conditions, batch_key='batch', batch_cond_key='sample_id')
dims = [8, 2]
base = scdisinfact(dictionary, Ks=dims, batch_size=64, lr=5e-4, seed=SEED, device=DEVICE)
losses = base.train_model(nepochs=BASE_EPOCHS - 1, recon_loss='NB')
base.eval()
torch.save({'state_dict': base.state_dict(), 'scaler': dictionary['scaler'],
            'matching_dict': dictionary['matching_dict'], 'dims': dims}, OUT / 'backbone.pt')
backbone = Backbone(base, data.conditions)
train_encoded = backbone.encode(training.counts, training.meta)
val_encoded = backbone.encode(validation.counts, validation.meta)
backbone.fit_centroids(train_encoded)
bandwidth = bandwidth_from_training(np.log1p(normalized(training.counts)), SEED)
plt.plot(np.arange(len(losses[0])) * 10, losses[0], marker='o')
plt.xlabel('Epoch'); plt.ylabel('Backbone diagnostic loss'); plt.show()
print('مدل اصلی آموزش دید و ذخیره شد.')


## ۶. نتیجهٔ مدل اصلی، قبل از ایدهٔ ما

هر روش با همان سلول‌های واقعی تست مقایسه می‌شود. `fixed` روش اصلی مقاله است. `no_shift` شرایط زیستی را تغییر نمی‌دهد. خطای کمتر بهتر است. سلول بعدی توابع ارزیابی و نمایش را تعریف می‌کند؛ خودِ آموزش انجام نمی‌دهد.


In [ ]:
jobs, skipped = scenarios(data, manifest, 'gbm', None, 50)
display(pd.DataFrame(skipped)) if skipped else print('تمام مسیرهای انتخاب‌شده قابل ارزیابی‌اند.')

def evaluate_methods(adapters=None):
    rows = []
    for name, smask, tmask, labels, batch in jobs:
        source, target = subset(data, smask), subset(data, tmask)
        enc = backbone.encode(source.counts, source.meta)
        target_enc = backbone.encode(target.counts, target.meta)
        ids = [names.index(value) for names, value in zip(backbone.labels, labels)]
        references = {'raw': normalized(target.counts),
                      'denoised': backbone.predict(target_enc, mode='identity')}
        predictions = {'no_shift': backbone.predict(enc, ids, batch, mode='identity'),
                       'fixed': backbone.predict(enc, ids, batch)}
        for mode, adapter in (adapters or {}).items():
            predictions[mode] = backbone.predict(enc, ids, batch, adapter=adapter)
        for method, prediction in predictions.items():
            for ct in sorted(target.meta.cell_type.unique()):
                si = source.meta.cell_type.eq(ct).values
                ti = target.meta.cell_type.eq(ct).values
                source_mean = normalized(source.counts[si]).mean(0)
                for reference, truth in references.items():
                    rows.append(dict(experiment='gbm', seed=SEED, scenario=name, cell_type=ct,
                                     method=method, reference=reference, n_source=int(si.sum()), n_target=int(ti.sum()),
                                     **metrics(prediction[si], truth[ti], source_mean, bandwidth, SEED)))
    return pd.DataFrame(rows)

def show_results(results):
    observed = results.loc[results.reference.eq('raw')]
    display(observed[['scenario', 'cell_type', 'method', 'mse', 'pearson', 'r2']].round(5))
    observed.pivot_table(index=['scenario', 'cell_type'], columns='method', values='mse').plot.bar(figsize=(12, 5))
    plt.ylabel('MSE against observed test cells (lower is better)')
    plt.tight_layout(); plt.show()


In [ ]:
baseline_results = evaluate_methods()
baseline_results.to_csv(OUT / 'baseline_metrics.csv', index=False)
show_results(baseline_results)


## ۷. اضافه‌کردن ایدهٔ تو

حالا مدل اصلی ثابت می‌ماند. دو روش جدید آموزش می‌دهیم:

- `constant`: یک جابه‌جایی قابل‌آموزش که برای همهٔ سلول‌ها یکسان است؛ کنترل مقایسه.
- `network`: شبکهٔ کوچک ایدهٔ تو که جابه‌جایی هر سلول را جدا می‌سازد.

سلول‌های تست همچنان وارد آموزش نمی‌شوند. وزن‌های مدل اصلی پس از آموزش دوباره بررسی می‌شوند تا تغییر نکرده باشند.


In [ ]:
adapters = {}
for mode in ['constant', 'network']:
    print('شروع آموزش:', mode, flush=True)
    adapter, history = train_adapter(backbone, train_encoded, val_encoded,
        training.meta, validation.meta, dims, mode, bandwidth, SEED,
        epochs=ADAPTER_EPOCHS, steps=STEPS)
    adapters[mode] = adapter
    torch.save({'state_dict': adapter.state_dict(), 'mode': mode, 'dims': dims,
                'categories': [len(x) for x in backbone.labels], 'bandwidth': bandwidth,
                'centroids': [x.cpu() for x in backbone.centroids]}, OUT / f'{mode}.pt')
    frame = pd.DataFrame(history)
    frame.to_csv(OUT / f'{mode}_history.csv', index=False)
    frame.plot(x='epoch', y=['train_mmd', 'val_mmd'], title=mode, figsize=(7, 3))
    plt.ylabel('Distribution matching loss'); plt.show()


## ۸. مقایسهٔ قبل و بعد

اکنون هر چهار روش را روی **همان تست** مقایسه می‌کنیم. مهم‌ترین مقایسه `network` در برابر `fixed` است. بهترشدن نسبت به `constant` کمک می‌کند بفهمیم آیا وابستگی به سلول واقعاً مفید بوده است.

این فقط یک seed و یک آزمایش است؛ هنوز نتیجهٔ نهایی کل پروژه نیست.


In [ ]:
results = evaluate_methods(adapters)
results.to_csv(OUT / 'metrics.csv', index=False)
show_results(results)
paired = results.loc[results.reference.eq('raw')].pivot(index=['scenario', 'cell_type'], columns='method', values='mse')
paired['network_improvement_percent'] = (paired['fixed'] - paired['network']) / paired['fixed'].replace(0, np.nan) * 100
display(paired.round(4))
print('درصد مثبت یعنی خطای کمتر از روش اصلی؛ درصد منفی یعنی بدترشدن.')
print('خروجی ذخیره شد در:', OUT)


## ۹. آزمایش‌های تکمیلی پلن ـ بعد از بررسی نتیجهٔ بالا

این قسمت اختیاری است و **لازم نیست الان اجرا کنی**. برای سه seed، آزمون حذف نوع سلول و شبیه‌سازی، از همان کدهای پروژه استفاده می‌کنیم؛ خروجی زنده در همین نوت‌بوک دیده می‌شود. اجرا ممکن است طولانی باشد. اجرای ناقص بازنویسی نمی‌شود.


In [ ]:
# فقط وقتی خواستیم مرحلهٔ تکمیلی را شروع کنیم، True کن.
RUN_EXTENDED = False
if RUN_EXTENDED:
    if not torch.cuda.is_available():
        raise RuntimeError('برای اجرای مجموعهٔ کامل، GPU لازم است.')
    run_visible(['cellshift.download', 'simulated'])
    for seed in [0, 1, 2]:
        for experiment in ['gbm', 'gbm_celltype', 'sim_is', 'sim_oos']:
            destination = ROOT / 'runs' / experiment / f'seed_{seed}'
            if (destination / 'timing.json').exists() and (destination / 'metrics.csv').exists():
                print('قبلاً کامل شده:', destination)
                continue
            run_visible(['cellshift.run', '--experiment', experiment, '--seed', str(seed), '--device', 'cuda'])
    run_visible(['cellshift.report'])
    from IPython.display import Image
    display(Image(filename=str(ROOT / 'runs' / 'comparison.png')))
    display(Markdown((ROOT / 'runs' / 'REPORT.md').read_text()))
else:
    print('آزمایش‌های تکمیلی هنوز شروع نشده‌اند.')
